In [1]:
import pandas as pd
import numpy as np

In [ ]:
df_encoded = pd.read_csv("../data/financial_fraud_detection_dataset.csv")

In [9]:
%whos

Variable   Type         Data/Info
---------------------------------
df         DataFrame    Shape: (5000000, 18)
np         module       <module 'numpy' from 'c:\<...>ges\\numpy\\__init__.py'>
pd         module       <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>


In [10]:
df_encoded = pd.read_csv("../data/financial_fraud_detection_dataset.csv")
print("Loaded successfully!")

Loaded successfully!


In [11]:
df_encoded.shape

(5000000, 18)

In [12]:
df_encoded.columns

Index(['transaction_id', 'timestamp', 'sender_account', 'receiver_account',
       'amount', 'transaction_type', 'merchant_category', 'location',
       'device_used', 'is_fraud', 'fraud_type', 'time_since_last_transaction',
       'spending_deviation_score', 'velocity_score', 'geo_anomaly_score',
       'payment_channel', 'ip_address', 'device_hash'],
      dtype='str')

In [13]:
df_encoded.isnull().sum()

transaction_id                       0
timestamp                            0
sender_account                       0
receiver_account                     0
amount                               0
transaction_type                     0
merchant_category                    0
location                             0
device_used                          0
is_fraud                             0
fraud_type                     4820447
time_since_last_transaction     896513
spending_deviation_score             0
velocity_score                       0
geo_anomaly_score                    0
payment_channel                      0
ip_address                           0
device_hash                          0
dtype: int64

In [14]:
df_encoded['fraud_type'] = df_encoded['fraud_type'].fillna('not_fraud')

df_encoded['time_missing_flag'] = (
    df_encoded['time_since_last_transaction']
    .isnull()
    .astype(int)
)

df_encoded['time_since_last_transaction'] = (
    df_encoded['time_since_last_transaction']
    .fillna(0)
)

In [15]:
df_encoded = df_encoded.drop(
    columns=[
        'transaction_id',
        'timestamp',
        'sender_account',
        'receiver_account',
        'ip_address',
        'device_hash'
    ]
)

In [16]:
df_encoded = pd.get_dummies(
    df_encoded,
    columns=[
        'transaction_type',
        'merchant_category',
        'location',
        'device_used',
        'payment_channel',
        'fraud_type'
    ],
    drop_first=True
)

In [17]:
X = df_encoded.drop('is_fraud', axis=1)
y = df_encoded['is_fraud']

In [18]:
X = X.drop('time_missing_flag', axis=1)

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
from xgboost import XGBClassifier

xgb_final = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_final.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = xgb_final.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      1.00      1.00    964089
        True       1.00      1.00      1.00     35911

    accuracy                           1.00   1000000
   macro avg       1.00      1.00      1.00   1000000
weighted avg       1.00      1.00      1.00   1000000

[[964089      0]
 [     0  35911]]


In [23]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_final.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(15))

                         Feature  Importance
28          fraud_type_not_fraud         1.0
1    time_since_last_transaction         0.0
2       spending_deviation_score         0.0
3                 velocity_score         0.0
0                         amount         0.0
5       transaction_type_payment         0.0
6      transaction_type_transfer         0.0
7    transaction_type_withdrawal         0.0
8      merchant_category_grocery         0.0
9       merchant_category_online         0.0
10       merchant_category_other         0.0
11  merchant_category_restaurant         0.0
4              geo_anomaly_score         0.0
12      merchant_category_retail         0.0
13      merchant_category_travel         0.0


In [24]:
df_encoded['time_since_last_transaction'] = (
    df_encoded['time_since_last_transaction'].fillna(0)
)

In [25]:
df_encoded = df_encoded.drop(
    columns=[
        'transaction_id',
        'timestamp',
        'sender_account',
        'receiver_account',
        'ip_address',
        'device_hash',
        'fraud_type'
    ]
)

KeyError: "['transaction_id', 'timestamp', 'sender_account', 'receiver_account', 'ip_address', 'device_hash', 'fraud_type'] not found in axis"

In [26]:
df_encoded.columns.tolist()

['amount',
 'is_fraud',
 'time_since_last_transaction',
 'spending_deviation_score',
 'velocity_score',
 'geo_anomaly_score',
 'time_missing_flag',
 'transaction_type_payment',
 'transaction_type_transfer',
 'transaction_type_withdrawal',
 'merchant_category_grocery',
 'merchant_category_online',
 'merchant_category_other',
 'merchant_category_restaurant',
 'merchant_category_retail',
 'merchant_category_travel',
 'merchant_category_utilities',
 'location_Dubai',
 'location_London',
 'location_New York',
 'location_Singapore',
 'location_Sydney',
 'location_Tokyo',
 'location_Toronto',
 'device_used_mobile',
 'device_used_pos',
 'device_used_web',
 'payment_channel_UPI',
 'payment_channel_card',
 'payment_channel_wire_transfer',
 'fraud_type_not_fraud']

In [27]:
df_encoded.shape

(5000000, 31)

In [28]:
df_model = df_encoded.drop(
    columns=['time_missing_flag', 'fraud_type_not_fraud']
)

In [29]:
X = df_model.drop('is_fraud', axis=1)
y = df_model['is_fraud']

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
from xgboost import XGBClassifier

xgb_final = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_final.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [32]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = xgb_final.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

              precision    recall  f1-score   support

       False       0.96      1.00      0.98    964089
        True       0.00      0.00      0.00     35911

    accuracy                           0.96   1000000
   macro avg       0.48      0.50      0.49   1000000
weighted avg       0.93      0.96      0.95   1000000

[[964089      0]
 [ 35911      0]]


In [33]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_final.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(15))

                        Feature  Importance
1   time_since_last_transaction    0.680457
8     merchant_category_grocery    0.013889
17            location_New York    0.013593
13     merchant_category_travel    0.013211
22           device_used_mobile    0.013022
24              device_used_web    0.012843
0                        amount    0.012781
12     merchant_category_retail    0.012755
5      transaction_type_payment    0.012445
25          payment_channel_UPI    0.012417
26         payment_channel_card    0.012319
4             geo_anomaly_score    0.012261
23              device_used_pos    0.012150
2      spending_deviation_score    0.012147
16              location_London    0.012115


In [34]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

              precision    recall  f1-score   support

       False       0.96      1.00      0.98    964089
        True       0.00      0.00      0.00     35911

    accuracy                           0.96   1000000
   macro avg       0.48      0.50      0.49   1000000
weighted avg       0.93      0.96      0.95   1000000

[[964089      0]
 [ 35911      0]]


In [35]:
from xgboost import XGBClassifier

xgb_balanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=26.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_balanced.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [36]:
y_pred = xgb_balanced.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      0.20      0.34    964089
        True       0.04      0.98      0.08     35911

    accuracy                           0.23   1000000
   macro avg       0.52      0.59      0.21   1000000
weighted avg       0.96      0.23      0.33   1000000

[[197222 766867]
 [   832  35079]]


In [37]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_final.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(15))

                        Feature  Importance
1   time_since_last_transaction    0.680457
8     merchant_category_grocery    0.013889
17            location_New York    0.013593
13     merchant_category_travel    0.013211
22           device_used_mobile    0.013022
24              device_used_web    0.012843
0                        amount    0.012781
12     merchant_category_retail    0.012755
5      transaction_type_payment    0.012445
25          payment_channel_UPI    0.012417
26         payment_channel_card    0.012319
4             geo_anomaly_score    0.012261
23              device_used_pos    0.012150
2      spending_deviation_score    0.012147
16              location_London    0.012115


In [38]:
X2 = X.drop('time_since_last_transaction', axis=1)

In [39]:
from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
xgb_test = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_test.fit(X_train2, y_train2)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [41]:
y_pred2 = xgb_test.predict(X_test2)

print(classification_report(y_test2, y_pred2))
print(confusion_matrix(y_test2, y_pred2))

c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

       False       0.96      1.00      0.98    964089
        True       0.00      0.00      0.00     35911

    accuracy                           0.96   1000000
   macro avg       0.48      0.50      0.49   1000000
weighted avg       0.93      0.96      0.95   1000000

[[964089      0]
 [ 35911      0]]


c:\Users\mkart\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [43]:
import pandas as pd

importance2 = pd.DataFrame({
    'Feature': X_train2.columns,
    'Importance': xgb_test.feature_importances_
})

importance2 = importance2.sort_values(
    by='Importance',
    ascending=False
)

print(importance2.head(15))

                         Feature  Importance
14                location_Dubai    0.042615
5      transaction_type_transfer    0.039528
22               device_used_pos    0.039280
16             location_New York    0.039175
10  merchant_category_restaurant    0.039160
6    transaction_type_withdrawal    0.039035
15               location_London    0.038706
1       spending_deviation_score    0.038608
18               location_Sydney    0.038370
0                         amount    0.037981
8       merchant_category_online    0.037845
11      merchant_category_retail    0.037733
3              geo_anomaly_score    0.037096
2                 velocity_score    0.037080
12      merchant_category_travel    0.036640


In [44]:
xgb_balanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=26.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_balanced.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [45]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_bal = xgb_balanced.predict(X_test)

print(classification_report(y_test, y_pred_bal))
print(confusion_matrix(y_test, y_pred_bal))

              precision    recall  f1-score   support

       False       1.00      0.20      0.34    964089
        True       0.04      0.98      0.08     35911

    accuracy                           0.23   1000000
   macro avg       0.52      0.59      0.21   1000000
weighted avg       0.96      0.23      0.33   1000000

[[197222 766867]
 [   832  35079]]


In [46]:
proba = xgb_balanced.predict_proba(X_test)[:,1]

In [47]:
import numpy as np

risk_level = np.where(
    proba >= 0.70,
    'High Risk Fraud',
    np.where(
        proba >= 0.30,
        'Suspicious',
        'Low Risk'
    )
)

In [48]:
results = pd.DataFrame({
    'Actual': y_test.values,
    'Fraud_Probability': proba,
    'Risk_Level': risk_level
})

results.head(20)

,Actual,Fraud_Probability,Risk_Level
0,False,0.026892,Low Risk
1,False,0.542116,Suspicious
2,False,0.016348,Low Risk
3,False,0.025491,Low Risk
4,False,0.540711,Suspicious
5,True,0.542043,Suspicious
6,False,0.557125,Suspicious
7,False,0.556711,Suspicious
8,False,0.559423,Suspicious
9,False,0.025930,Low Risk


In [49]:
def explain_transaction(row):
    reasons = []

    if row['amount'] > 1000:
        reasons.append('High transaction amount')

    if row['velocity_score'] > 15:
        reasons.append('High transaction velocity')

    if row['geo_anomaly_score'] > 0.8:
        reasons.append('Unusual location activity')

    if row['spending_deviation_score'] > 1:
        reasons.append('Abnormal spending pattern')

    return ', '.join(reasons) if reasons else 'No major anomalies detected'

In [50]:
results['Reason'] = X_test.apply(explain_transaction, axis=1)

In [51]:
sample = X_test.iloc[[0]]
sample

,amount,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,transaction_type_payment,transaction_type_transfer,transaction_type_withdrawal,merchant_category_grocery,merchant_category_online,...,location_Singapore,location_Sydney,location_Tokyo,location_Toronto,device_used_mobile,device_used_pos,device_used_web,payment_channel_UPI,payment_channel_card,payment_channel_wire_transfer
743476,21.4,0.0,-0.72,4,0.27,False,True,False,False,False,...,True,False,False,False,False,True,False,False,False,False


In [56]:
proba = xgb_balanced.predict_proba(sample)[0][1]
prediction = xgb_balanced.predict(sample)[0]

print("Prediction:", prediction)
print("Fraud Probability:", round(proba, 4))

if proba >= 0.70:
    risk = "🔴 High Risk Fraud"
elif proba >= 0.30:
    risk = "🟡 Suspicious"
else:
    risk = "🟢 Low Risk"

print("Risk Level:", risk)

reasons = []

row = sample.iloc[0]

if row['amount'] > X['amount'].quantile(0.90):
    reasons.append("High transaction amount")

if row['velocity_score'] > X['velocity_score'].quantile(0.90):
    reasons.append("High transaction velocity")

if row['geo_anomaly_score'] > X['geo_anomaly_score'].quantile(0.90):
    reasons.append("Unusual geographical activity")

if row['spending_deviation_score'] > X['spending_deviation_score'].quantile(0.90):
    reasons.append("Abnormal spending behavior")

print("Reasons:")
if reasons:
    for r in reasons:
        print("-", r)
else:
    print("- No major anomalies detected")

Prediction: 0
Fraud Probability: 0.0269
Risk Level: 🟢 Low Risk
Reasons:
- No major anomalies detected


In [57]:
y_prob = xgb_balanced.predict_proba(X_test)[:, 1]

In [58]:
y_pred = (y_prob >= 0.5).astype(int)

In [59]:
fraud_cases = X_test[
    (X_test['location_Dubai'] == False) &
    (y_pred == 1)
]

fraud_cases.head()

,amount,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,transaction_type_payment,transaction_type_transfer,transaction_type_withdrawal,merchant_category_grocery,merchant_category_online,...,location_Singapore,location_Sydney,location_Tokyo,location_Toronto,device_used_mobile,device_used_pos,device_used_web,payment_channel_UPI,payment_channel_card,payment_channel_wire_transfer
1588704,39.63,4984.368321,0.73,8,0.97,False,True,False,False,False,...,True,False,False,False,False,False,True,False,False,True
3426921,37.89,-247.098577,0.26,10,0.67,False,False,True,False,True,...,False,False,False,False,False,False,False,True,False,False
2641995,116.21,-789.622257,0.04,4,0.99,False,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False
3881897,47.80,-4004.885514,0.20,17,0.81,False,True,False,False,False,...,False,False,True,False,False,False,True,False,True,False
1847310,120.38,-1588.262720,0.24,18,0.04,False,False,True,False,False,...,False,False,False,True,False,False,True,False,False,False


In [128]:
# ==========================================
# FRAUD ANALYSIS REPORT
# ==========================================

txn_no = int(input("Enter transaction number: "))

# Check if transaction exists
if txn_no not in X.index:
    print("❌ Transaction not found .")

else:
    # -------------------------
    # Get transaction
    # -------------------------
    sample = X.loc[[txn_no]]
    row = sample.iloc[0]

    # -------------------------
    # Fraud Probability
    # -------------------------
    prob = xgb_balanced.predict_proba(sample)[0][1]

    # -------------------------
    # Transaction Location
    # -------------------------
    location_cols = [
        c for c in sample.columns
        if c.startswith('location_')
    ]

    transaction_location = "Reference Location"

    for col in location_cols:
        if row[col]:
            transaction_location = col.replace(
                'location_',
                ''
            )
            break

    # -------------------------
    # Simulated Home Location
    # -------------------------
    customer_home_location = "Chennai"

    location_mismatch = (
        customer_home_location != transaction_location
    )

    # -------------------------
    # Transaction Type
    # -------------------------
    transaction_type = "Unknown"

    if row.get('transaction_type_payment', False):
        transaction_type = "Payment"
    elif row.get('transaction_type_transfer', False):
        transaction_type = "Transfer"
    elif row.get('transaction_type_withdrawal', False):
        transaction_type = "Withdrawal"

    # -------------------------
    # Payment Channel
    # -------------------------
    payment_channel = "Unknown"

    if row.get('payment_channel_UPI', False):
        payment_channel = "UPI"
    elif row.get('payment_channel_card', False):
        payment_channel = "Card"
    elif row.get('payment_channel_wire_transfer', False):
        payment_channel = "Wire Transfer"

    # -------------------------
    # Device Used
    # -------------------------
    device = "Unknown"

    if row.get('device_used_mobile', False):
        device = "Mobile"
    elif row.get('device_used_web', False):
        device = "Web"
    elif row.get('device_used_pos', False):
        device = "POS"

    # -------------------------
    # Risk Classification
    # -------------------------
    amount = row['amount']

    if amount < 100:
        risk = "🟢 Low Risk"
        status = "NOT FLAGGED"
        recommendation = (
            "Low-value transaction (< ₹100). "
            "No fraud escalation required."
        )

    else:
        if prob >= 0.60:
            risk = "🔴 High Risk Fraud"
            status = "FLAGGED"
            recommendation = (
                "Temporarily hold transaction "
                "and send for fraud investigation."
            )

        elif prob >= 0.30:
            risk = "🟡 Suspicious"
            status = "REVIEW REQUIRED"
            recommendation = (
                "Perform additional verification "
                "(OTP / customer confirmation)."
            )

        else:
            risk = "🟢 Low Risk"
            status = "NOT FLAGGED"
            recommendation = (
                "Approve transaction."
            )

    # -------------------------
    # Reasons
    # -------------------------
    reasons = []

    if amount > X['amount'].quantile(0.90):
        reasons.append(
            "High transaction amount"
        )

    if row['velocity_score'] > X['velocity_score'].quantile(0.90):
        reasons.append(
            "High transaction velocity"
        )

    if row['geo_anomaly_score'] > X['geo_anomaly_score'].quantile(0.90):
        reasons.append(
            "Unusual geographical activity"
        )

    if abs(row['spending_deviation_score']) > 1:
        reasons.append(
            "Abnormal spending behaviour"
        )

    if location_mismatch:
        reasons.append(
            f"Transaction made from "
            f"{transaction_location} while "
            f"customer's usual location is "
            f"{customer_home_location}"
        )

    if row.get('payment_channel_wire_transfer', False):
        reasons.append(
            "Wire transfer transactions are "
            "considered higher risk"
        )

    if row.get('transaction_type_transfer', False):
        reasons.append(
            "Money transfer transaction"
        )

    # -------------------------
    # Final Report
    # -------------------------
    print("\n")
    print("=" * 50)
    print("          FRAUD ANALYSIS REPORT")
    print("=" * 50)

    print(f"Transaction Number      : {txn_no}")
    print(f"Transaction Amount      : ₹{amount:,.2f}")

    print(
        f"Customer Home Location  : "
        f"{customer_home_location}"
    )

    print(
        f"Transaction Location    : "
        f"{transaction_location}"
    )

    print(
        f"Location Mismatch       : "
        f"{location_mismatch}"
    )

    print(
        f"Transaction Type        : "
        f"{transaction_type}"
    )

    print(
        f"Payment Channel         : "
        f"{payment_channel}"
    )

    print(
        f"Device Used             : "
        f"{device}"
    )

    print(
        f"Velocity Score          : "
        f"{row['velocity_score']:.2f}"
    )

    print(
        f"Geo Anomaly Score       : "
        f"{row['geo_anomaly_score']:.2f}"
    )

    print(
        f"Spending Deviation      : "
        f"{row['spending_deviation_score']:.2f}"
    )

    print(
        f"\nFraud Probability       : "
        f"{prob:.2%}"
    )

    print(f"Risk Level              : {risk}")
    print(f"Status                  : {status}")

    print("\nWhy was this decision made?")

if risk == "🟢 Low Risk":

    print("• The overall fraud probability is low.")
    print("• Although some unusual characteristics were observed,")
    print("  they were not strong enough in combination to indicate fraud.")
    print("• The transaction behaviour is largely consistent with")
    print("  normal transaction patterns learned by the model.")

elif risk == "🟡 Suspicious":

    print("• The transaction exhibits some risk indicators:")

    for r in reasons:
        print("  -", r)

    print("• Additional verification is recommended.")

else:

    print("• The transaction exhibits multiple high-risk characteristics:")

    for r in reasons:
        print("  -", r)

    print("• The combined evidence resulted in a high fraud probability.")



          FRAUD ANALYSIS REPORT
Transaction Number      : 3942363
Transaction Amount      : ₹2,315.23
Customer Home Location  : Chennai
Transaction Location    : Tokyo
Location Mismatch       : True
Transaction Type        : Unknown
Payment Channel         : Unknown
Device Used             : Web
Velocity Score          : 18.00
Geo Anomaly Score       : 0.02
Spending Deviation      : 0.43

Fraud Probability       : 73.48%
Risk Level              : 🔴 High Risk Fraud
Status                  : FLAGGED

Why was this decision made?
• The transaction exhibits multiple high-risk characteristics:
  - High transaction amount
  - Transaction made from Tokyo while customer's usual location is Chennai
• The combined evidence resulted in a high fraud probability.


In [113]:
df.loc[3942363, 'is_fraud']
df.loc[1555434, 'is_fraud']
df.loc[2770601, 'is_fraud']

np.False_

In [108]:
txn_no = 3942363

sample = X_test.loc[[txn_no]]

In [109]:
raw_txn = df.loc[txn_no]
print(raw_txn)

transaction_id                                   T4042363
timestamp                      2023-08-05T04:18:56.236892
sender_account                                  ACC553874
receiver_account                                ACC243270
amount                                            2315.23
transaction_type                                  deposit
merchant_category                                  online
location                                            Tokyo
device_used                                           web
is_fraud                                            False
fraud_type                                            NaN
time_since_last_transaction                  -1880.617687
spending_deviation_score                             0.43
velocity_score                                         18
geo_anomaly_score                                    0.02
payment_channel                                       ACH
ip_address                                230.143.248.246
device_hash   

In [114]:
probs = xgb_balanced.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    'Actual': y_test.values,
    'Probability': probs
})

In [116]:
high_risk_actual_frauds = results[
    (results['Actual'] == True) &
    (results['Probability'] >= 0.60)
]

len(high_risk_actual_frauds)

93

In [105]:
probs = xgb_balanced.predict_proba(X_test)[:, 1]

high_risk_idx = X_test.index[probs >= 0.70]

high_risk = list(zip(
    high_risk_idx,
    probs[probs >= 0.70]
))

high_risk[:10]

[(1555434, np.float32(0.7003254)),
 (3942363, np.float32(0.73479784)),
 (2770601, np.float32(0.7023965))]